# Topic 17E — Dates and Calculated Columns

Dates are powerful after pandas understands them as dates. Calculated columns let us enrich a clean dataset with useful new information.

**Main lesson link:** [Topic 17 — Data Cleaning with pandas](topic17_data_cleaning.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

filename = "data_cleaning_with_pandas/data/students_messy.csv"
possible_paths = [Path(filename), Path("data") / filename, Path("../data") / filename, Path("../../data") / filename]
data_path = next(path for path in possible_paths if path.exists())
df = pd.read_csv(data_path)
print("Loaded:", data_path)
print(df.shape)


---
## 1. Inspect Date Strings


In [ ]:
print(df["date_enrolled"].head(15).to_string())
print("dtype before:", df["date_enrolled"].dtype)
print("Dates containing slash:", df["date_enrolled"].str.contains("/", na=False).sum())


---
## 2. Convert with `pd.to_datetime()`

`errors="coerce"` turns unparseable dates into `NaT`, pandas' missing datetime value.


In [ ]:
df["date_enrolled_clean"] = pd.to_datetime(df["date_enrolled"], dayfirst=True, errors="coerce")

print("dtype after:", df["date_enrolled_clean"].dtype)
print("Failed parses:", df["date_enrolled_clean"].isna().sum())
print(df[["date_enrolled", "date_enrolled_clean"]].head(10))


---
## 3. Extract Date Parts


In [ ]:
df["enrol_year"] = df["date_enrolled_clean"].dt.year
df["enrol_month"] = df["date_enrolled_clean"].dt.month

df[["date_enrolled_clean", "enrol_year", "enrol_month"]].head()


---
## 4. Calculated Score Columns


In [ ]:
# Minimal score cleaning for this example
df["math_score_num"] = pd.to_numeric(df["math_score"].astype(str).str.replace("%", "", regex=False), errors="coerce")
df.loc[(df["math_score_num"] < 0) | (df["math_score_num"] > 100), "math_score_num"] = np.nan
for col in ["math_score_num", "english_score", "science_score"]:
    df[col] = df[col].fillna(df[col].median())

df["avg_score"] = df[["math_score_num", "english_score", "science_score"]].mean(axis=1).round(1)
df["result"] = np.where(df["avg_score"] >= 50, "Pass", "Fail")

print(df[["student_id", "avg_score", "result"]].head())


---
## 5. Analyse with the New Columns


In [ ]:
print("Pass counts:")
print(df["result"].value_counts())
print()
print("Average score by enrolment year:")
print(df.groupby("enrol_year")["avg_score"].mean().round(1))


### Useful Resources

- [pandas API: `to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas user guide: Time series / date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [pandas getting started: Create new columns](https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html)


---
## Quick Reflection

1. What is `NaT`?
2. Why should dates be converted before date calculations?
3. What is one useful calculated column you could create from student scores?
